# 02 — Data Collection

**Objective:** Systematically gather evidence for all 30 vehicle-part combinations from public online sources.

**Key rule:** One row in `raw_observations.csv` = one source observation of one vehicle-part application. If two suppliers list the same brake pad, record both — they are separate observations that may corroborate each other.

**Assignment requirement:** *"Record every source URL and collection date. Use at least two sources where possible."*

In [1]:
import pandas as pd
import os

# Load source log
source_log = pd.read_csv('../data/source_log.csv')
print(f"Registered sources: {len(source_log)}")
source_log

Registered sources: 8


,source_id,source_name,url,source_type,collection_date,what_it_provides,reliability_note
0,S001,Adey Market,https://adey-market.com/categories/cars?type=c...,Marketplace,2026-11-18,Vehicle marketplace listing counts and pricing...,Listings represent advertised seller inventory...
1,S002,Gezat Electric,https://ev.gezat.et/,Distributor catalogue,2026-11-18,Commercial EV model availability in the Ethiop...,Investigated at catalogue/model level ONLY — e...
2,S003,Landin BYD Parts,https://www.landinparts.com/byd/,Parts supplier catalogue,2026-11-18,BYD spare parts with supplier-listed OE number...,Supplier catalogue — zero Yuan Up listings con...
3,S004,CQSW Auto Parts Benchmark,https://www.cqswautoparts.com/part-numbers-for...,Parts reference,2026-11-18,BYD Seagull part number reference benchmark us...,Secondary reference benchmark — used only to c...
4,S005,toyota-spare-parts.com (Third-Party Catalogue),https://www.toyota-spare-parts.com/toyota-cars...,Third-party parts catalogue,2026-11-18,Toyota bZ4X part numbers and prices; NOT an of...,Third-party aggregator catalogue — part number...
5,S006,NHTSA Datasets and APIs,https://www.nhtsa.gov/nhtsa-datasets-and-apis,Government safety database,2026-11-18,US-market vehicle defect complaints and recall...,US-market data only — provides engineering pla...
6,S007,AutoZone,https://www.autozone.com,Commercial aftermarket catalogue,2026-11-18,Aftermarket consumable fitment cross-checks fo...,Aftermarket catalogue reference — consumable f...
7,S008,Unverified Trade Quotation (Yuan Up),unverified_trade_quotation — URL not recorded ...,Trade quotation,2026-11-18,Price and fitment claims for BYD Yuan Up parts...,UNVERIFIED — Part numbers are trade listing co...


## 2.1 Data Collection Checklist

For each of the 30 vehicle-part combinations, check:

1. ✅ Does a compatible part exist in the catalogue?
2. ✅ Is an OEM part number available?
3. ✅ Can you find a supplier or listing?
4. ✅ Is a price visible?
5. ✅ Is availability stated?
6. ✅ Can another source confirm the information?

### Status values to use:
- `found` — Part listed with evidence
- `not_listed` — Searched the source but found no listing
- `not_verifiable` — A listing exists but compatibility cannot be confirmed
- `not_accessible` — The page could not be accessed
- Leave blank — Haven't investigated yet

## 2.2 Collection Process

Each of the 30 vehicle-part combinations was searched against the registered sources. Observations were recorded in `data/raw/raw_observations.csv`, one row per source observation.

**Sources searched, by vehicle:**

| Vehicle | Sources searched | Notes |
|---------|-----------------|-------|
| BYD Seagull | S003 (landinparts.com), S004 (cqswautoparts.com benchmark) | Full 10-category sweep; 2 catalogue gaps recorded (OBS036, OBS037) |
| BYD Yuan Up | S003 (landinparts.com — zero Yuan Up listings), S008 (unverified export trade quotation) | No authoritative parts catalogue; 6 catalogue gaps + 4 unverified trade quotations |
| Toyota bZ4X | S005 (toyota-spare-parts.com), S006 (NHTSA safety database) | Full 10-category sweep; 2 NHTSA safety-context rows (OBS058, OBS059) |

> The part-by-part evidence for every combination is generated directly from `raw_observations.csv` in the §2.3 cell below, so the table cannot drift from the dataset.

In [2]:
# After collecting data, load and check progress
import pandas as pd
from pathlib import Path

_candidates = [Path.cwd(), *Path.cwd().parents]
_root = next(p for p in _candidates if (p / 'data' / 'raw' / 'raw_observations.csv').exists())
raw_data = pd.read_csv(_root / 'data' / 'raw' / 'raw_observations.csv')
print(f"Total observations collected: {len(raw_data)}")
print()

if len(raw_data) > 0:
    # Check coverage: how many of the 30 combinations have at least one observation?
    coverage = raw_data.groupby(['vehicle', 'part_category']).size().reset_index(name='observation_count')
    print(f"Vehicle-part combinations with data: {len(coverage)} / 30")
    print()
    
    # Show which combinations are still missing
    vehicles = ['BYD Seagull', 'BYD Yuan Up', 'Toyota bZ4X']
    parts = ['Brake pads', 'Brake discs', 'Cabin-air filter', 'Wiper blades',
             'Headlight assembly', 'Tail-light assembly', 'Side mirror',
             'Bumper', 'Fender / body panel', 'Shock absorber']
    
    all_combos = pd.DataFrame([(v, p) for v in vehicles for p in parts],
                              columns=['vehicle', 'part_category'])
    merged = all_combos.merge(coverage, on=['vehicle', 'part_category'], how='left')
    missing = merged[merged['observation_count'].isna()]
    
    if len(missing) > 0:
        print(f"Combinations still needing investigation ({len(missing)}):")
        print(missing[['vehicle', 'part_category']].to_string(index=False))
    else:
        print("All 30 combinations have at least one observation!")
else:
    print("No data collected yet. Start adding rows to data/raw/raw_observations.csv")

Total observations collected: 59

Vehicle-part combinations with data: 30 / 30

All 30 combinations have at least one observation!


## 2.3 Collection Notes — Part-by-Part Evidence

The table below is generated directly from `data/raw/raw_observations.csv`. For every vehicle-part combination it lists the observation IDs, the sources searched, the observation types, and the collection date. Combinations with no observation row are reported explicitly as **NOT INVESTIGATED** rather than left blank.

> **Status:** all 30 combinations have at least one recorded observation, so no combination is marked *not investigated*. Genuinely missing evidence is itemised in `reports/limitations.md` (e.g. no per-part search was captured for the Gezat distributor catalogue, S002).

In [3]:
import pandas as pd
from pathlib import Path

_candidates = [Path.cwd(), *Path.cwd().parents]
_root = next(p for p in _candidates if (p / 'data' / 'raw' / 'raw_observations.csv').exists())
raw_data = pd.read_csv(_root / 'data' / 'raw' / 'raw_observations.csv')
print('Raw observations on file:', len(raw_data))

VEHICLES = ['BYD Seagull', 'BYD Yuan Up', 'Toyota bZ4X']
PARTS = ['Brake pads', 'Brake discs', 'Cabin-air filter', 'Wiper blades',
         'Headlight assembly', 'Tail-light assembly', 'Side mirror',
         'Bumper', 'Fender / body panel', 'Shock absorber']

records = []
for v in VEHICLES:
    for p in PARTS:
        sub = raw_data[(raw_data['vehicle'] == v) & (raw_data['part_category'] == p)]
        if len(sub) == 0:
            records.append({'vehicle': v, 'part_category': p, 'observations': 0,
                            'observation_ids': '—', 'sources': '—',
                            'observation_types': 'NOT INVESTIGATED (no evidence row collected)',
                            'collection_date': '—'})
        else:
            records.append({
                'vehicle': v,
                'part_category': p,
                'observations': len(sub),
                'observation_ids': ', '.join(sub['observation_id']),
                'sources': ', '.join(sorted(set(sub['source_id'].astype(str)))),
                'observation_types': ' / '.join(sorted(set(sub['observation_type'].astype(str)))),
                'collection_date': ', '.join(sorted(set(sub['collection_date'].astype(str)))),
            })

evidence_table = pd.DataFrame(records)
covered = int((evidence_table['observations'] > 0).sum())
print('Combinations with at least one observation:', covered, 'of 30')
for v in VEHICLES:
    block = evidence_table[evidence_table['vehicle'] == v].drop(columns='vehicle')
    print()
    print('--- ' + v + ' ---')
    print(block.to_string(index=False))

evidence_table


Raw observations on file: 59
Combinations with at least one observation: 30 of 30

--- BYD Seagull ---
      part_category  observations                                                                                                                observation_ids sources  observation_types collection_date
         Brake pads             2                                                                                                                 OBS024, OBS025    S003 supplier_catalogue      2026-11-18
        Brake discs             2                                                                                                                 OBS034, OBS035    S003 supplier_catalogue      2026-11-18
   Cabin-air filter             1                                                                                                                         OBS036    S003      catalogue_gap      2026-11-18
       Wiper blades             1                                                

,vehicle,part_category,observations,observation_ids,sources,observation_types,collection_date
0,BYD Seagull,Brake pads,2,"OBS024, OBS025",S003,supplier_catalogue,2026-11-18
1,BYD Seagull,Brake discs,2,"OBS034, OBS035",S003,supplier_catalogue,2026-11-18
2,BYD Seagull,Cabin-air filter,1,OBS036,S003,catalogue_gap,2026-11-18
3,BYD Seagull,Wiper blades,1,OBS037,S003,catalogue_gap,2026-11-18
4,BYD Seagull,Headlight assembly,2,"OBS013, OBS014",S003,supplier_catalogue,2026-11-18
5,BYD Seagull,Tail-light assembly,3,"OBS015, OBS016, OBS017",S003,supplier_catalogue,2026-11-18
6,BYD Seagull,Side mirror,2,"OBS019, OBS020",S003,supplier_catalogue,2026-11-18
7,BYD Seagull,Bumper,5,"OBS001, OBS002, OBS003, OBS004, OBS028",S003,supplier_catalogue,2026-11-18
8,BYD Seagull,Fender / body panel,16,"OBS005, OBS006, OBS007, OBS008, OBS009, OBS010...",S003,supplier_catalogue,2026-11-18
9,BYD Seagull,Shock absorber,3,"OBS021, OBS022, OBS023",S003,supplier_catalogue,2026-11-18


## 2.4 Adding New Sources

If you discover a new source during collection, add it to `source_log.csv` before recording observations from it:

In [4]:
# Example: Adding a new source (uncomment and modify when needed)
# new_source = pd.DataFrame([{
#     'source_id': 'S009',
#     'source_name': 'Example Supplier',
#     'url': 'https://example.com',
#     'source_type': 'Parts supplier catalogue',
#     'collection_date': '2026-11-18',
#     'what_it_provides': 'Part pricing and availability',
#     'reliability_note': 'Verify fitment details'
# }])
# 
# source_log = pd.read_csv('../data/source_log.csv')
# source_log = pd.concat([source_log, new_source], ignore_index=True)
# source_log.to_csv('../data/source_log.csv', index=False)
# print("Source added successfully")